# Predicting Cognitive Stress and Fatigue with Keystroke Behavior
**Milestone 1 — Baseline Model & Statistical Analysis**

**Course:** Statistical and Machine Learning  
**Team:** Graham Phillips  
**Date:** March 2026

---

## Overview
This notebook covers:
1. Data loading (Kaggle stress dataset + IKDD keystroke dynamics)
2. Feature engineering from raw keystroke events
3. Exploratory Data Analysis (EDA)
4. Statistical Analysis (distributions, hypothesis testing, bootstrapping)
5. Sliding window data augmentation
6. Baseline Logistic Regression model
7. Model evaluation

## 0. Imports & Configuration

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

# ── PATH CONFIGURATION ──────────────────────────────────────────────────────
KAGGLE_PATH = "/Users/grahamphillips/Downloads/KeyStroke"   # folder with 'user 1', 'user 2'
IKDD_PATH   = "/Users/grahamphillips/IKDD/IKDD"             # folder with .txt files
USERS       = ["user 1", "user 2"]
# ─────────────────────────────────────────────────────────────────────────────
print("Setup complete.")

---
## 1. Load Kaggle Stress Dataset

In [ ]:
def load_user_data(base_path, user):
    """Load all TSV files for a given user."""
    user_path = os.path.join(base_path, user)
    ks  = pd.read_csv(os.path.join(user_path, "keystrokes.tsv"),      sep="\t")
    uc  = pd.read_csv(os.path.join(user_path, "usercondition.tsv"),   sep="\t")
    md  = pd.read_csv(os.path.join(user_path, "mousedata.tsv"),       sep="\t")
    ks["user"]  = user
    uc["user"]  = user
    return ks, uc, md

all_ks, all_uc = [], []
for u in USERS:
    ks, uc, md = load_user_data(KAGGLE_PATH, u)
    all_ks.append(ks)
    all_uc.append(uc)

keystrokes_df = pd.concat(all_ks, ignore_index=True)
condition_df  = pd.concat(all_uc, ignore_index=True)

# Parse timestamps
keystrokes_df["Press_Time"]   = pd.to_datetime(keystrokes_df["Press_Time"])
keystrokes_df["Relase_Time"]  = pd.to_datetime(keystrokes_df["Relase_Time"])
condition_df["Time"]          = pd.to_datetime(condition_df["Time"])

print(f"Keystrokes : {keystrokes_df.shape}")
print(f"Conditions : {condition_df.shape}")
keystrokes_df.head(3)

In [ ]:
condition_df.head(10)

### 1.1 Create Binary Stress Label
We simplify `Stress_Val` into a binary label: **Stressed** (S_Stressed or V_Stressed) vs **Not Stressed** (Neutral, F_Good).

In [ ]:
def binarize_stress(val):
    if pd.isna(val):
        return np.nan
    return 1 if "Stressed" in str(val) else 0

condition_df["stress_label"] = condition_df["Stress_Val"].apply(binarize_stress)
condition_df["fatigue_label"] = condition_df["Fatigue_Val"].apply(
    lambda x: 1 if str(x) in ["Low", "Below_Avg"] else 0
)

print("Stress label distribution:")
print(condition_df["stress_label"].value_counts())
print("\nFatigue label distribution:")
print(condition_df["fatigue_label"].value_counts())

---
## 2. Feature Engineering from Keystroke Events

For each time window (aligned to a `usercondition` label), we compute:
- **Dwell time** — how long a key is held (`Release_Time - Press_Time`)
- **Inter-key interval (IKI)** — time between consecutive key presses
- **Typing speed** — keys per second
- **Error rate** — proportion of backspace/delete keys
- **Rhythm variability** — coefficient of variation (CV) of IKI

In [ ]:
def extract_features(ks_window):
    """Extract behavioral features from a window of keystroke events."""
    if len(ks_window) < 5:
        return None

    # Dwell time (ms)
    dwell = (ks_window["Relase_Time"] - ks_window["Press_Time"]).dt.total_seconds() * 1000
    dwell = dwell[dwell > 0]

    # Inter-key interval (ms)
    iki = ks_window["Press_Time"].diff().dt.total_seconds().dropna() * 1000
    iki = iki[(iki > 0) & (iki < 5000)]  # remove outliers >5s

    # Typing speed (keys per second)
    duration = (ks_window["Press_Time"].max() - ks_window["Press_Time"].min()).total_seconds()
    speed = len(ks_window) / duration if duration > 0 else 0

    # Error rate (backspace/delete)
    error_keys = ks_window["Key"].str.lower().isin(["backspace", "delete"])
    error_rate = error_keys.sum() / len(ks_window)

    features = {
        "dwell_mean":    dwell.mean()   if len(dwell) > 0 else np.nan,
        "dwell_std":     dwell.std()    if len(dwell) > 0 else np.nan,
        "iki_mean":      iki.mean()     if len(iki) > 0 else np.nan,
        "iki_std":       iki.std()      if len(iki) > 0 else np.nan,
        "iki_cv":        (iki.std() / iki.mean()) if (len(iki) > 0 and iki.mean() > 0) else np.nan,
        "typing_speed":  speed,
        "error_rate":    error_rate,
        "n_keys":        len(ks_window),
    }
    return features

print("Feature extraction function defined.")

### 2.1 Assign Labels via Nearest Timestamp Join
Each keystroke window is labeled using the nearest `usercondition` entry (backward fill within 30 min).

In [ ]:
def build_feature_dataset(keystrokes_df, condition_df, window_minutes=10):
    """
    Slide a window over keystrokes and extract features,
    then join with the nearest condition label.
    """
    records = []
    
    for user in keystrokes_df["user"].unique():
        ks_user   = keystrokes_df[keystrokes_df["user"] == user].sort_values("Press_Time").copy()
        uc_user   = condition_df[condition_df["user"] == user].sort_values("Time").copy()
        
        if ks_user.empty or uc_user.empty:
            continue

        # Sliding window: step = window_minutes / 2  (50% overlap)
        start_time = ks_user["Press_Time"].min()
        end_time   = ks_user["Press_Time"].max()
        window_td  = pd.Timedelta(minutes=window_minutes)
        step_td    = pd.Timedelta(minutes=window_minutes / 2)

        t = start_time
        while t + window_td <= end_time:
            window = ks_user[
                (ks_user["Press_Time"] >= t) &
                (ks_user["Press_Time"] <  t + window_td)
            ]
            feats = extract_features(window)
            if feats is None:
                t += step_td
                continue

            # Find nearest preceding condition label
            prior = uc_user[uc_user["Time"] <= t + window_td]
            if prior.empty:
                t += step_td
                continue
            nearest = prior.iloc[-1]

            feats["stress_label"]  = nearest["stress_label"]
            feats["fatigue_label"] = nearest["fatigue_label"]
            feats["user"]          = user
            feats["window_start"]  = t
            records.append(feats)
            t += step_td

    return pd.DataFrame(records)

feature_df = build_feature_dataset(keystrokes_df, condition_df, window_minutes=10)
feature_df = feature_df.dropna()

print(f"Feature dataset shape: {feature_df.shape}")
print(f"Stress label counts:\n{feature_df['stress_label'].value_counts()}")
feature_df.head()

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
feature_cols = ["dwell_mean", "dwell_std", "iki_mean", "iki_std", "iki_cv", "typing_speed", "error_rate"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    for label, grp in feature_df.groupby("stress_label"):
        axes[i].hist(grp[col].dropna(), bins=20, alpha=0.6,
                     label=f"{'Stressed' if label==1 else 'Not Stressed'}")
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle("Feature Distributions by Stress Label", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(9, 7))
corr = feature_df[feature_cols + ["stress_label"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots: key features by stress label
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ["iki_mean", "typing_speed", "error_rate"]):
    feature_df.boxplot(column=col, by="stress_label", ax=ax)
    ax.set_title(f"{col} by Stress")
    ax.set_xlabel("Stress Label (0=No, 1=Yes)")
plt.suptitle("")
plt.tight_layout()
plt.show()

---
## 4. Statistical Analysis

### 4.1 Hypothesis Testing
We use the **Mann-Whitney U test** (non-parametric) to test whether each feature differs significantly between stressed and non-stressed windows.

**H₀:** The feature distribution is the same across stress groups.  
**H₁:** The distributions differ.

In [ ]:
stressed     = feature_df[feature_df["stress_label"] == 1]
not_stressed = feature_df[feature_df["stress_label"] == 0]

print(f"{'Feature':<20} {'U-stat':>10} {'p-value':>12} {'Significant?':>14}")
print("-" * 60)
for col in feature_cols:
    u_stat, p_val = stats.mannwhitneyu(
        stressed[col].dropna(),
        not_stressed[col].dropna(),
        alternative='two-sided'
    )
    sig = "✓ Yes" if p_val < 0.05 else "No"
    print(f"{col:<20} {u_stat:>10.1f} {p_val:>12.4f} {sig:>14}")

### 4.2 Distribution Fitting
We check whether inter-key interval (IKI) follows a log-normal or exponential distribution — a common assumption in keystroke dynamics literature.

In [ ]:
iki_vals = feature_df["iki_mean"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram with fitted distributions
axes[0].hist(iki_vals, bins=30, density=True, alpha=0.6, label="Observed")

# Fit log-normal
shape, loc, scale = stats.lognorm.fit(iki_vals, floc=0)
x = np.linspace(iki_vals.min(), iki_vals.max(), 200)
axes[0].plot(x, stats.lognorm.pdf(x, shape, loc, scale), 'r-', lw=2, label="Log-Normal fit")

# Fit normal
mu, sigma = stats.norm.fit(iki_vals)
axes[0].plot(x, stats.norm.pdf(x, mu, sigma), 'g--', lw=2, label="Normal fit")
axes[0].set_title("IKI Mean — Distribution Fit")
axes[0].set_xlabel("IKI Mean (ms)")
axes[0].legend()

# Q-Q plot for normality
stats.probplot(iki_vals, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot of IKI Mean")

plt.tight_layout()
plt.show()

# Shapiro-Wilk normality test
stat, p = stats.shapiro(iki_vals.sample(min(len(iki_vals), 200), random_state=42))
print(f"Shapiro-Wilk: stat={stat:.4f}, p={p:.4f}")
print("IKI Mean appears", "normal" if p > 0.05 else "non-normal", f"(p={'> 0.05' if p > 0.05 else '< 0.05'})")

### 4.3 Bootstrapping — Confidence Intervals for Group Means
We use bootstrapping to estimate 95% confidence intervals for the mean IKI in stressed vs. non-stressed groups.

In [ ]:
def bootstrap_ci(data, n_boot=2000, ci=0.95, stat_fn=np.mean, seed=42):
    """Return (lower, upper) bootstrap confidence interval."""
    rng = np.random.default_rng(seed)
    boot_stats = [
        stat_fn(rng.choice(data, size=len(data), replace=True))
        for _ in range(n_boot)
    ]
    alpha = (1 - ci) / 2
    return np.quantile(boot_stats, alpha), np.quantile(boot_stats, 1 - alpha)

print("Bootstrap 95% CIs for IKI Mean:")
for label, name in [(0, "Not Stressed"), (1, "Stressed")]:
    vals = feature_df[feature_df["stress_label"] == label]["iki_mean"].dropna().values
    lo, hi = bootstrap_ci(vals)
    print(f"  {name:15s}: mean={vals.mean():.2f} ms  95% CI=[{lo:.2f}, {hi:.2f}]")

In [ ]:
# Visualize bootstrap distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
rng = np.random.default_rng(42)

for ax, (label, name) in zip(axes, [(0, "Not Stressed"), (1, "Stressed")]):
    vals = feature_df[feature_df["stress_label"] == label]["iki_mean"].dropna().values
    boot_means = [
        np.mean(rng.choice(vals, size=len(vals), replace=True))
        for _ in range(2000)
    ]
    lo, hi = np.quantile(boot_means, 0.025), np.quantile(boot_means, 0.975)
    ax.hist(boot_means, bins=40, color="steelblue" if label == 0 else "tomato", alpha=0.75)
    ax.axvline(lo, color='black', linestyle='--', label=f"95% CI [{lo:.1f}, {hi:.1f}]")
    ax.axvline(hi, color='black', linestyle='--')
    ax.set_title(f"Bootstrap Distribution of Mean IKI — {name}")
    ax.set_xlabel("Mean IKI (ms)")
    ax.legend()

plt.tight_layout()
plt.show()

---
## 5. IKDD Dataset — Behavioral Feature Extraction
The IKDD dataset provides additional unlabeled keystroke sessions from many users. We extract the same features to understand behavioral variation across users and sessions.

In [ ]:
def parse_ikdd_file(filepath):
    """
    Parse one IKDD .txt file.
    Line 1: metadata (userID, gender, age, handedness, language, education, device, experience)
    Remaining lines: key_code-finger_code, [inter-key intervals in ms...]
    Returns a flat list of inter-key intervals and metadata.
    """
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.read().splitlines()

    if not lines:
        return None

    # Parse metadata
    meta = lines[0].split(',')
    user_id = meta[0] if len(meta) > 0 else "unknown"
    gender  = meta[1] if len(meta) > 1 else "unknown"
    age     = meta[2] if len(meta) > 2 else "unknown"

    # Parse IKI values
    all_ikis = []
    for line in lines[1:]:
        parts = line.strip().split(',')
        if len(parts) < 2:
            continue
        try:
            ikis = [float(v) for v in parts[1:] if v.strip()]
            all_ikis.extend(ikis)
        except ValueError:
            continue

    if len(all_ikis) < 10:
        return None

    ikis = np.array(all_ikis)
    ikis = ikis[(ikis > 0) & (ikis < 2000)]  # filter outliers

    return {
        "user_id":      user_id,
        "gender":       gender,
        "age":          age,
        "iki_mean":     ikis.mean(),
        "iki_std":      ikis.std(),
        "iki_cv":       ikis.std() / ikis.mean() if ikis.mean() > 0 else np.nan,
        "iki_median":   np.median(ikis),
        "n_keystrokes": len(ikis),
        "filename":     os.path.basename(filepath),
    }

# Load all IKDD files
ikdd_files = glob.glob(os.path.join(IKDD_PATH, "*.txt"))
print(f"Found {len(ikdd_files)} IKDD files.")

ikdd_records = [parse_ikdd_file(f) for f in ikdd_files]
ikdd_df = pd.DataFrame([r for r in ikdd_records if r is not None])
print(f"IKDD feature dataset: {ikdd_df.shape}")
ikdd_df.head()

In [ ]:
# IKI distribution across IKDD users
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(ikdd_df["iki_mean"].dropna(), bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("IKDD: Distribution of Mean IKI Across Sessions")
axes[0].set_xlabel("Mean IKI (ms)")

if "gender" in ikdd_df.columns:
    for g, grp in ikdd_df.groupby("gender"):
        axes[1].hist(grp["iki_mean"].dropna(), bins=20, alpha=0.6, label=str(g))
    axes[1].set_title("IKDD: Mean IKI by Gender")
    axes[1].set_xlabel("Mean IKI (ms)")
    axes[1].legend()

plt.tight_layout()
plt.show()

print(ikdd_df[["iki_mean", "iki_std", "iki_cv"]].describe())

---
## 6. Baseline Model — Logistic Regression

We train a Logistic Regression classifier on the engineered features from the Kaggle dataset to predict binary stress labels.

In [ ]:
FEATURE_COLS = ["dwell_mean", "dwell_std", "iki_mean", "iki_std", "iki_cv", "typing_speed", "error_rate"]
TARGET = "stress_label"

model_df = feature_df[FEATURE_COLS + [TARGET]].dropna()
X = model_df[FEATURE_COLS].values
y = model_df[TARGET].values.astype(int)

print(f"Samples: {len(X)}  |  Class balance: {np.bincount(y)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr",     LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("\n── Classification Report ──")
print(classification_report(y_test, y_pred, target_names=["Not Stressed", "Stressed"]))

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Not Stressed", "Stressed"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title("Confusion Matrix — Logistic Regression")

# Feature coefficients
coefs = pipeline.named_steps["lr"].coef_[0]
axes[1].barh(FEATURE_COLS, coefs, color=["tomato" if c > 0 else "steelblue" for c in coefs])
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Logistic Regression Coefficients")
axes[1].set_xlabel("Coefficient Value")

plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc")

print(f"5-Fold Cross-Validation AUC-ROC:")
print(f"  Scores : {cv_scores.round(3)}")
print(f"  Mean   : {cv_scores.mean():.3f}")
print(f"  Std    : {cv_scores.std():.3f}")

# Bootstrap CI on test AUC
auc_test = roc_auc_score(y_test, y_prob)
print(f"\nTest AUC-ROC: {auc_test:.3f}")

# Bootstrap confidence interval on CV AUC
lo, hi = bootstrap_ci(cv_scores, n_boot=2000)
print(f"Bootstrap 95% CI on CV AUC: [{lo:.3f}, {hi:.3f}]")

---
## 7. Summary

| Component | Result |
|---|---|
| Dataset (Kaggle) | 2 users, keystroke + condition labels |
| Dataset (IKDD) | ~100+ user sessions, raw keystroke timing |
| Features engineered | Dwell time, IKI mean/std/CV, typing speed, error rate |
| Augmentation | Sliding window (10 min, 50% overlap) |
| Statistical tests | Mann-Whitney U, Shapiro-Wilk, Bootstrap CI |
| Baseline model | Logistic Regression (standardized features, balanced classes) |
| Evaluation | 5-fold CV AUC-ROC + Bootstrap CI |

### Next Steps (Milestone 2)
- Expand to Random Forest, SVM, Gradient Boosted Trees
- Hyperparameter tuning
- Incorporate mouse behavior features
- Build Streamlit real-time inference app